# Data Ingestion

In [1]:
from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader

def laod_data(directory_path):
    data=DirectoryLoader(directory_path,glob="*.pdf",loader_cls=PyPDFLoader)
    docs=data.load()

    return docs


    

C:\Users\USER\AppData\Local\Temp\ipykernel_9060\1211247238.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader
d:\HR-Policy-RAG-Chatbot\venkat\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
policy_docs=laod_data("D:/HR-Policy-RAG-Chatbot/data")
policy_docs

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Windows)', 'creationdate': '2023-01-27T09:40:38+05:30', 'moddate': '2023-01-27T09:40:47+05:30', 'trapped': '/False', 'source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf', 'total_pages': 208, 'page': 0, 'page_label': 'a'}, page_content='a\nIIMA HR Policy Manual 2023\nHUMAN RESOURCES  \nPOLICY MANUAL\nSTAFF \n2023'),
 Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Windows)', 'creationdate': '2023-01-27T09:40:38+05:30', 'moddate': '2023-01-27T09:40:47+05:30', 'trapped': '/False', 'source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf', 'total_pages': 208, 'page': 1, 'page_label': 'b'}, page_content='b\nIIMA HR Policy Manual 2023'),
 Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Windows)', 'creationdate': '2023-01-27T09:40:38+05:30', 'moddate': '2023-01-27T09:40:47+05:30', '

In [20]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(documents:List[Document]) -> List[Document]:
    '''Given a list of documents Objects, return a new list of documents with only the page content and source metadata.'''
    minimal_docs=[]
    for doc in policy_docs:
        src=doc.metadata.get('source')
        minimal_docs.append(Document(page_content=doc.page_content,metadata={'Source':src}))
    return minimal_docs

       


In [21]:
filter_docs=filter_to_minimal_docs(policy_docs)
filter_docs 

[Document(metadata={'Source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf'}, page_content='a\nIIMA HR Policy Manual 2023\nHUMAN RESOURCES  \nPOLICY MANUAL\nSTAFF \n2023'),
 Document(metadata={'Source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf'}, page_content='b\nIIMA HR Policy Manual 2023'),
 Document(metadata={'Source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf'}, page_content='i\nIIMA HR Policy Manual 2023\nDECLARATION\nThe objective of this Manual is to compile  the HR policies and \nprocedures followed in IIMA. It also presents the general rules and \nregulations  that govern the employees of the Institute. \nThis Manual supersedes all previous manuals, handbooks, and \nmemorandums that may have been issued from time to time on \nsubjects covered in this Manual.\nThe Institute reserves its right to interpret; change; suspend; cancel; \nor dispute, with or without notice; all or any part of what is contained \nin the M

# Clean the Text

In [23]:
import re 
import string 
from typing import List
from langchain_core.documents import Document

def clean_text(docs:List[Document]) -> Document:
    clean_docs=[]

    for doc in docs:
        # text=re.sub(r'\s+','',doc.page_content).strip()   # Remove extra whitespace and newlines
        text=re.sub(r'[^\w\s\.\,\-\!\?\;\:]','',doc.page_content)     #  # Remove special characters but keep basic punctuation

        clean_docs.append(Document(page_content=text,metadata=doc.metadata))

    return clean_docs


cleaned_text=clean_text(filter_docs)  

cleaned_text



[Document(metadata={'Source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf'}, page_content='a\nIIMA HR Policy Manual 2023\nHUMAN RESOURCES  \nPOLICY MANUAL\nSTAFF \n2023'),
 Document(metadata={'Source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf'}, page_content='b\nIIMA HR Policy Manual 2023'),
 Document(metadata={'Source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf'}, page_content='i\nIIMA HR Policy Manual 2023\nDECLARATION\nThe objective of this Manual is to compile  the HR policies and \nprocedures followed in IIMA. It also presents the general rules and \nregulations  that govern the employees of the Institute. \nThis Manual supersedes all previous manuals, handbooks, and \nmemorandums that may have been issued from time to time on \nsubjects covered in this Manual.\nThe Institute reserves its right to interpret; change; suspend; cancel; \nor dispute, with or without notice; all or any part of what is contained \nin the M

# Chunk the text

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter



def chunks_text(docs:List[Document]) -> Document:

    split_text=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=40)
    text_splits=split_text.split_documents(cleaned_text)

    return text_splits


chunks=chunks_text(cleaned_text)  

chunks



[Document(metadata={'Source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf'}, page_content='a\nIIMA HR Policy Manual 2023\nHUMAN RESOURCES  \nPOLICY MANUAL\nSTAFF \n2023'),
 Document(metadata={'Source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf'}, page_content='b\nIIMA HR Policy Manual 2023'),
 Document(metadata={'Source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf'}, page_content='i\nIIMA HR Policy Manual 2023\nDECLARATION'),
 Document(metadata={'Source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf'}, page_content='DECLARATION\nThe objective of this Manual is to compile  the HR policies and'),
 Document(metadata={'Source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf'}, page_content='procedures followed in IIMA. It also presents the general rules and'),
 Document(metadata={'Source': 'D:\\HR-Policy-RAG-Chatbot\\data\\HR Policy Manual 2023 (8).pdf'}, page_content='regulations  that gover

In [25]:
from langchain_huggingface import HuggingFaceEmbeddings


def download_embeddings():
    '''Download and initialize HuggingFace embeddings model'''
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embedding_model=download_embeddings()
embedding_model

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2154.14it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [26]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone

load_dotenv()

PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")

pc=Pinecone(api_key=PINECONE_API_KEY)

pc

In [27]:
print(pc.list_indexes().names())


['medical-chatbot', 'hr-policy-chatbot']


In [28]:
from pinecone import Pinecone, ServerlessSpec
index_name = "hr-policy-chatbot"


if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [29]:
# we have created an index in pinecone, now we need to add our data to the index. We will use the embedding model to create embeddings for our chunks of text and then we will add those embeddings to the index.

from langchain_pinecone import PineconeVectorStore

docsearch=PineconeVectorStore.from_documents(documents=chunks,embedding=embedding_model,index_name=index_name)

docsearch

In [30]:
user_query="Can casual leave be accumulated if not used?"

retriever=docsearch.as_retriever(search_type="similarity",search_kwargs={"k":5})

retriever_docs=retriever.invoke(user_query)

context=" ".join([ doc.page_content for doc in retriever_docs])

context

'condition that not more than five days casual leave may be allowed at a time. 5.1.2 Casual leave can be combined with Special Casual leave but not with any other kind of \nleave. 5.1.7 Casual leave cannot be accumulated. Leave not availed in a particular calendar year the accumulated casual leave in the last month results in disruption of work. 5.1.5 Casual Leave can be taken while on tour, but no daily allowance will be admissible for'

In [31]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an HR policy assistant.

Use ONLY the context below.

If the answer is not in the context, say:
"I could not find this in the HR policy documents provided."

Context:
{context}

Question:
{user_query}

Answer:
""")

In [32]:
import os 
from dotenv import load_dotenv

load_dotenv()


groq_api_key=os.getenv("groq_api")


from langchain_groq import ChatGroq

llm=ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=groq_api_key,
    temperature=0.3,
    max_retries=3

)

message = prompt.invoke({"user_query": user_query, "context": context})
response=llm.invoke(message)

response.content

'No, casual leave cannot be accumulated.'